In [4]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import wsb_utils as wu


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=43373, as_frame=True, parser="auto").frame
print(df.shape)
df.columns

(53187, 7)


Index(['title', 'score', 'url', 'comms_num', 'created', 'body', 'timestamp'],
      dtype='str')

In [6]:
url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt"
sym = pd.read_csv(url, sep="|")
valid_tickers = set(sym["Symbol"].dropna().astype(str))
print(len(valid_tickers), "symbols")

13285 symbols


In [7]:
start = wu.first_full_day(df, date_col="created")
print("Data starts properly on:", start.date())

days = wu.to_trading_day(df["created"])
df_full = df[days.values >= start]
print(len(df_full), "posts kept of", len(df))

mentions = wu.daily_mentions(df_full, valid_tickers, date_col="created")
spikes = wu.find_spikes(mentions)
print(len(spikes), "spikes")
spikes.head(20)

Data starts properly on: 2021-01-28
53186 posts kept of 53187
48 spikes


,date,ticker,mentions,avg_before
0,2021-02-11,TLRY,21,2.500000
1,2021-02-19,PLTR,37,7.142857
2,2021-02-22,PLTR,49,9.357143
3,2021-03-01,RKT,18,2.642857
4,2021-03-02,RKT,40,3.642857
5,2021-03-03,RKT,185,6.428571
6,2021-03-03,UWMC,15,0.571429
7,2021-03-04,SKT,13,0.357143
8,2021-03-04,UWMC,33,1.642857
9,2021-03-08,HF,10,1.785714


In [8]:
# the date range of the posts and of the spikes
dates = wu.to_trading_day(df["created"])
print("Posts:", dates.min().date(), "→", dates.max().date())
print("Spikes:", spikes["date"].min().date(), "→", spikes["date"].max().date())

#most mentioned symbols overall: look for English words that slipped through
print(mentions.groupby("ticker")["mentions"].sum().sort_values(ascending=False).head(40))

# 3. Which symbols spike most
print(spikes["ticker"].value_counts().head(20))

Posts: 2020-09-29 → 2021-08-16
Spikes: 2021-02-11 → 2021-08-04
ticker
GME     12055
AMC      4485
BB       1866
NOK      1364
RH       1323
PLTR      752
RKT       715
TSLA      551
TD        417
SPY       413
LINE      403
UWMC      382
AMD       348
SNDL      338
CLOV      308
SPCE      274
HF        228
SI        224
AAPL      213
NIO       204
SLV       200
TIME      185
TLRY      175
RSI       170
CLNE      169
CRSR      166
CLF       164
HERE      164
AAL       156
MVIS      149
AMZN      147
WKHS      144
GM        143
PSA       139
ASO       129
LOT       125
KNOW      124
BABA      118
MA        109
DKNG      107
Name: mentions, dtype: int64
ticker
UWMC    6
RKT     5
PLTR    4
BB      3
CLOV    3
AMC     3
GME     3
SPCE    3
CLF     3
TSLA    2
MVIS    2
CLNE    2
WKHS    2
TLRY    1
SKT     1
HF      1
DKNG    1
SPY     1
SOFI    1
AMD     1
Name: count, dtype: int64


In [ ]:
results = []
for ratio in [3, 4, 5]:
    for min_m in [5, 8, 10]:
        n = len(wu.find_spikes(mentions, ratio=ratio, min_mentions=min_m))
        results.append({"ratio": ratio, "min_mentions": min_m, "spikes": n})
pd.DataFrame(results).pivot(index="ratio", columns="min_mentions", values="spikes")